In [1]:
#kyuki abhi hamare pass ek hi single table hai or isliye humko pehle pdf ke according normalize karna hoga matlab- ek hi table ko se 2 tables banani hogi taki JOIN practice ho sake.

In [2]:
#customers table banao(unique customers ka summary):
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')

customers_df = pd.read_sql("""
    SELECT DISTINCT "Customer ID", Country
    FROM transactions
    WHERE "Customer ID" IS NOT NULL
""", conn)

customers_df.to_sql('customers', conn, if_exists='replace', index=False)
print(customers_df.shape)

(5955, 2)


In [3]:
#Orders table banao(invoice-level summary):

In [4]:
orders_df = pd.read_sql("""
    SELECT Invoice, "Customer ID", MIN(InvoiceDate) as InvoiceDate,
            SUM(Quantity * Price) as OrderValue
    FROM transactions
    WHERE "Customer ID" IS NOT NULL
    GROUP BY Invoice, "Customer ID"
""", conn)

orders_df.to_sql('orders', conn, if_exists='replace', index=False)
print(orders_df.shape)


(44876, 4)


In [5]:
#INNER JOIN practice 

In [6]:
#customers or orders ko combine akro
q1 = pd.read_sql("""
    SELECT c."Customer ID", c.Country, o.Invoice, o.OrderValue
    FROM customers c
    INNER JOIN orders o ON c."Customer ID" = o."Customer ID"
    LIMIT 10
""", conn)

print(q1)

   Customer ID         Country  Invoice  OrderValue
0      13085.0  United Kingdom   489434      505.30
1      13085.0  United Kingdom   489435      145.80
2      13085.0  United Kingdom   490068      284.30
3      13085.0  United Kingdom   490069      161.40
4      13085.0  United Kingdom   496092      430.20
5      13085.0  United Kingdom   496166      490.20
6      13085.0  United Kingdom   544306      278.10
7      13085.0  United Kingdom   558996      137.98
8      13085.0  United Kingdom  C527339     -830.12
9      13085.0  United Kingdom  C551464     -143.70


In [7]:
#har customer ka total order value (JOIN + GROUP BY combine)
q2 = pd.read_sql("""
    SELECT c."Customer ID", c.Country, SUM(o.OrderValue) as total_value, COUNT(o.Invoice) as total_orders
    FROM customers c
    INNER JOIN orders o ON c."Customer ID" = o."Customer ID"
    GROUP BY c."Customer ID", c.Country
    ORDER BY total_value DESC
    LIMIT 10
""", conn)
print(q2)

   Customer ID         Country  total_value  total_orders
0      18102.0  United Kingdom    598215.22           153
1      14646.0     Netherlands    523342.07           164
2      14156.0            EIRE    296564.69           202
3      14911.0            EIRE    270248.53           510
4      17450.0  United Kingdom    233579.39            61
5      13694.0  United Kingdom    190825.52           164
6      17511.0  United Kingdom    171885.98            85
7      12415.0       Australia    143269.29            33
8      16684.0  United Kingdom    141502.25            65
9      15061.0  United Kingdom    136391.48           138


In [8]:
#Germany ke customers ke orders (JOIN + WHERE)
q3 = pd.read_sql("""
    SELECT c."Customer ID", c.Country, o.Invoice, o.InvoiceDate, o.OrderValue
    FROM customers c
    INNER JOIN orders o ON c."Customer ID" = o."Customer ID"
    WHERE c.Country = 'Germany'
    ORDER BY o.OrderValue DESC
    LIMIT 10
""", conn)
print(q3)

   Customer ID  Country Invoice          InvoiceDate  OrderValue
0      12590.0  Germany  552978  2011-05-12 14:46:00    9341.260
1      12477.0  Germany  564856  2011-08-31 09:11:00    4257.060
2      12497.0  Germany  530799  2010-11-04 12:31:00    3993.640
3      12709.0  Germany  505335  2010-04-21 13:00:00    3857.200
4      12472.0  Germany  537201  2010-12-05 14:19:00    3262.600
5      12481.0  Germany  504146  2010-04-11 13:10:00    2969.600
6      12709.0  Germany  501545  2010-03-17 14:59:00    2917.760
7      12709.0  Germany  516131  2010-07-16 15:26:00    2903.710
8      12671.0  Germany  504332  2010-04-12 16:30:00    2622.481
9      12477.0  Germany  530970  2010-11-05 09:35:00    2570.770


In [9]:
#INNER JOIN sirf wo rows dets hai jinka match dono tables me ho - kyuki customers table orders se hi banaya gya hai, yaha sab match ho jaiga.

In [10]:
conn.close()